# Phase 7: Evaluation and Documentation

## Customer Churn Prediction with Explainable AI

This notebook consolidates the findings from Phases 1–6 into a final evaluation summary. It loads the artifacts already produced (trained models, test sets, SHAP outputs) and presents the project's conclusions in one place, with the supporting numbers re-displayed directly from saved results.

## 1. Objective

Predict customer churn for a SaaS business using account, subscription, feature usage, and support ticket data (RavenStack SaaS Subscription and Churn Analysis dataset, Kaggle), and explain model predictions using SHAP.

**Headline finding:** the project surfaced and diagnosed two distinct forms of data leakage. Once both were corrected, the available features were found to have **no meaningful predictive power for churn** in this dataset (ROC-AUC ≈ 0.49–0.52 across three model families). This notebook documents that finding and the evidence behind it.


In [2]:
import pandas as pd
import numpy as np
import joblib
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix

# Load the models and test sets saved at the end of Phase 5
rf_model_account = joblib.load('../models/rf_model_account.pkl')
rf_model_row = joblib.load('../models/rf_model_row.pkl')

X_test_account = pd.read_csv('../data/X_test_account.csv')
y_test_account = pd.read_csv('../data/y_test_account.csv').squeeze()

X_test_row = pd.read_csv('../data/X_test_row.csv')
y_test_row = pd.read_csv('../data/y_test_row.csv').squeeze()

print("Loaded models and test sets successfully.")
print("Account-level test set:", X_test_account.shape)
print("Row-level test set:", X_test_row.shape)


Loaded models and test sets successfully.
Account-level test set: (100, 23)
Row-level test set: (1018, 32)


## 2. Data Leakage: Diagnosis and Fixes

### 2a. Feature leakage

The merged dataset (`final_integrated_dataset.csv`) included columns derived from `churn_events.csv`, which **only contains rows for accounts that had already churned**. Columns such as `churn_flag_x`, `churn_flag_y`, `is_active`, `total_churn_events`, `any_reactivation`, `total_refunds`, `any_preceding_upgrade`, `any_preceding_downgrade`, and `top_reason` were, directly or indirectly, restatements of the target variable itself.

### 2b. Group leakage

After removing the feature-leakage columns, the model still scored a perfect ROC-AUC of 1.0. A shallow (depth-3) decision tree scored only 0.731 by comparison — too large a gap to be explained by model complexity alone. Investigation showed that **500 unique accounts span 5,000 subscription rows**, and several features are account-level rather than subscription-level. A standard random train/test split let subscriptions from the same account appear in both sets, allowing the model to memorize account identities rather than generalize.

**Fix:** switched to `GroupShuffleSplit`, grouped by `account_id`.

### Summary of the diagnostic progression


In [4]:
progression = pd.DataFrame([
    {"Stage": "Initial model (raw features, random split)", "ROC-AUC": 1.000, "Issue": "Feature leakage (churn_events.csv derivatives)"},
    {"Stage": "After removing leaky columns (random split)", "ROC-AUC": 1.000, "Issue": "Group leakage (accounts split across train/test)"},
    {"Stage": "After GroupShuffleSplit (row-level)", "ROC-AUC": 0.522, "Issue": "None — honest baseline"},
    {"Stage": "Account-level aggregation (Random Forest)", "ROC-AUC": 0.495, "Issue": "None — honest baseline"},
    {"Stage": "Account-level + engineered ratio features (Random Forest)", "ROC-AUC": 0.485, "Issue": "None — ratio features did not improve signal"},
    {"Stage": "Account-level, Logistic Regression", "ROC-AUC": 0.437, "Issue": "Confirms weak signal is not model-specific"},
    {"Stage": "Account-level, Gradient Boosting", "ROC-AUC": 0.365, "Issue": "Confirms weak signal across a third model family"},
    {"Stage": "Account-level, 5-fold grouped CV (Random Forest)", "ROC-AUC": 0.446, "Issue": "Mean of folds ranging 0.41-0.55 — consistent with no real signal"},
])
progression


,Stage,ROC-AUC,Issue
0,"Initial model (raw features, random split)",1.000,Feature leakage (churn_events.csv derivatives)
1,After removing leaky columns (random split),1.000,Group leakage (accounts split across train/test)
2,After GroupShuffleSplit (row-level),0.522,None — honest baseline
3,Account-level aggregation (Random Forest),0.495,None — honest baseline
4,Account-level + engineered ratio features (Ran...,0.485,None — ratio features did not improve signal
5,"Account-level, Logistic Regression",0.437,Confirms weak signal is not model-specific
6,"Account-level, Gradient Boosting",0.365,Confirms weak signal across a third model family
7,"Account-level, 5-fold grouped CV (Random Forest)",0.446,Mean of folds ranging 0.41-0.55 — consistent w...


## 3. Final Model Evaluation

Re-generating the classification report and ROC-AUC for both final (leak-free) models, directly from the saved test sets, so this notebook's output is reproducible rather than copy-pasted.

### 3a. Account-level model

In [5]:
y_pred_account = rf_model_account.predict(X_test_account)
y_proba_account = rf_model_account.predict_proba(X_test_account)[:, 1]

print("=== Account-Level Model: Classification Report ===")
print(classification_report(y_test_account, y_pred_account))
print("Confusion Matrix:")
print(confusion_matrix(y_test_account, y_pred_account))
print("\nROC-AUC Score:", roc_auc_score(y_test_account, y_proba_account))


=== Account-Level Model: Classification Report ===
              precision    recall  f1-score   support

       False       0.00      0.00      0.00        30
        True       0.70      0.99      0.82        70

    accuracy                           0.69       100
   macro avg       0.35      0.49      0.41       100
weighted avg       0.49      0.69      0.57       100

Confusion Matrix:
[[ 0 30]
 [ 1 69]]

ROC-AUC Score: 0.48500000000000004


### 3b. Row-level model (grouped split)

In [6]:
y_pred_row = rf_model_row.predict(X_test_row)
y_proba_row = rf_model_row.predict_proba(X_test_row)[:, 1]

print("=== Row-Level Model: Classification Report ===")
print(classification_report(y_test_row, y_pred_row))
print("Confusion Matrix:")
print(confusion_matrix(y_test_row, y_pred_row))
print("\nROC-AUC Score:", roc_auc_score(y_test_row, y_proba_row))

=== Row-Level Model: Classification Report ===
              precision    recall  f1-score   support

       False       0.10      0.01      0.02       329
        True       0.67      0.95      0.78       689

    accuracy                           0.65      1018
   macro avg       0.38      0.48      0.40      1018
weighted avg       0.48      0.65      0.54      1018

Confusion Matrix:
[[  4 325]
 [ 36 653]]

ROC-AUC Score: 0.5224235820381947


## 4. Explainability (SHAP) — Summary

Full SHAP analysis was performed in `phase6_explainability.ipynb`. Key plots are reproduced below for reference.

**Account-level model:** SHAP values cluster tightly near zero (x-axis range ≈ -0.05 to 0.075) with mixed, non-separated coloring across nearly all features — the visual signature of a model with no meaningful feature to lean on.

**Row-level model:** a wider SHAP value range (≈ -0.10 to 0.10), with a few one-hot categorical features (`referral_source_partner`, `country_UK`, `industry_HealthTech`) showing a visually dramatic split. This is most likely **small-subgroup overfitting** — one-hot features split cleanly by color regardless of true predictive value, and the instability of the 5-fold CV scores (0.41-0.55) confirms these splits do not generalize.

![Account-level SHAP summary](../outputs/shap_summary_account.png)

![Row-level SHAP summary](../outputs/shap_summary_row.png)


## 5. Conclusion

Across two data granularities, three model families, and five independent diagnostic checks (shallow-tree comparison, grouped cross-validation, raw feature correlation, recency-trend analysis, and multi-model comparison), the available features in this dataset do not meaningfully predict churn once both feature leakage and group leakage are removed.

This is a legitimate, well-evidenced finding rather than a failed model. The core value of this project is the diagnostic process: catching a suspicious perfect score, tracing it to its root cause in the raw source tables, fixing it, catching a *second* independent leak the same way, and confirming the honest result through multiple, methodologically distinct checks — rather than accepting an inflated number at face value.

### What this project demonstrates
- End-to-end pipeline design across five merged data sources
- Systematic detection of feature leakage and group leakage — two distinct and commonly-confused failure modes
- Use of shallow-tree analysis, grouped cross-validation, and multi-level correlation checks as independent diagnostic tools
- Explainability (SHAP) applied and correctly interpreted, including recognizing spurious-looking patterns from small subgroups
- Honest reporting of a negative/weak result, with full methodological justification
